# Sanitary Ware Data Analysis and Model Preparation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
df = pd.read_csv("sanitary_ware_final_reclassified.csv")
df.head()

## 3. Initial Inspection

In [ ]:
df.info()
df.describe(include="all")
df.nunique()

## 4. Data Quality Checks

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df[df["Price_EGP"] <= 0]

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()
for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].str.strip()
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["Price_EGP"]).reset_index(drop=True)
df_clean.head()

## 6. Quality Level Analysis

In [ ]:
df_clean.groupby("Quality_Level")["Price_EGP"].describe()

In [ ]:
quality_price = df_clean.groupby("Quality_Level", as_index=False)["Price_EGP"].mean()
quality_price["Quality_Level"] = pd.Categorical(quality_price["Quality_Level"], categories=["Low","Medium","High"], ordered=True)
quality_price = quality_price.sort_values("Quality_Level")
sns.barplot(data=quality_price, x="Quality_Level", y="Price_EGP")
plt.title("Average Price by Quality Level")
plt.show()

## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df_clean["Price_EGP"], bins=30, kde=True)
plt.title("Price Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df_clean, x="Quality_Level", y="Price_EGP", order=["Low","Medium","High"])
plt.title("Price Distribution by Quality Level")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=df_clean, x="Subcategory", y="Price_EGP", estimator="mean", order=df_clean.groupby("Subcategory")["Price_EGP"].mean().sort_values(ascending=False).index)
plt.xticks(rotation=45)
plt.title("Average Price by Subcategory")
plt.show()

## 8. Outlier Analysis

In [ ]:
Q1 = df_clean["Price_EGP"].quantile(0.25)
Q3 = df_clean["Price_EGP"].quantile(0.75)
IQR = Q3-Q1
lower_bound = Q1-1.5*IQR
upper_bound = Q3+1.5*IQR
outliers = df_clean[(df_clean["Price_EGP"]<lower_bound)|(df_clean["Price_EGP"]>upper_bound)]
outliers.head()

## 9. Feature Engineering

In [ ]:
df_model = df_clean.copy()
df_model["Finishing_Category"] = "Sanitary Ware"
df_model["Estimated_Quantity"] = df_model["Rule_Value"]
df_model["Estimated_Total_Cost"] = df_model["Estimated_Quantity"] * df_model["Price_EGP"]
df_model.head()

## 10. Apartment Requirement Logic

In [ ]:
def calculate_plumbing_requirements(apartment_area, rooms, bathrooms=1):
    return {"Apartment_Area_m2": apartment_area, "Rooms": rooms, "Bathrooms": bathrooms}

## 11. Cost Estimation

In [ ]:
def estimate_plumbing_cost(data, apartment_area, bathrooms, quality_level=None):
    filtered = data.copy()
    if quality_level is not None:
        filtered = filtered[filtered["Quality_Level"] == quality_level]
    quantities = filtered["Rule_Value"].copy()
    mask = filtered["Quantity_Rule"].astype(str).str.contains("Bathroom", case=False, na=False)
    quantities.loc[mask] = filtered.loc[mask, "Rule_Value"] * bathrooms
    return (quantities * filtered["Price_EGP"]).sum()

## 12. Budget-Based Recommendation

In [ ]:
def recommend_plumbing_products(data, budget, quality_level, application=None):
    filtered = data[data["Quality_Level"] == quality_level].copy()
    if application is not None:
        filtered = filtered[filtered["Application"].astype(str).str.contains(str(application), case=False, na=False)]
    filtered = filtered[filtered["Price_EGP"] <= budget]
    return filtered.sort_values("Price_EGP")

## 13. Multi-File Model Integration Structure

In [ ]:
common_columns = ["Finishing_Category","Category","Subcategory","Product_Name","Brand","Quality_Level","Price_EGP","Unit","Quantity_Rule","Rule_Value","Required_For","Optional"]
plumbing_for_master_model = df_model[[c for c in common_columns if c in df_model.columns]].copy()
plumbing_for_master_model.head()

## 14. Prepare Data for Future Master Model

In [ ]:
model_ready_data = plumbing_for_master_model.dropna(subset=["Price_EGP"]).reset_index(drop=True)
model_ready_data.info()

## 15. Final Validation and Export

In [ ]:
model_ready_data.to_csv("sanitary_ware_model_ready_reclassified.csv", index=False)
model_ready_data.head()

## 16. Final Project Role

This dataset will be integrated with the other finishing categories in the Master Product Dataset. The prepared category data will later be used to generate apartment-level scenarios for training the final scikit-learn model.